In [23]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict
import json
# 加载 .env 文件中的环境变量
load_dotenv()

True

# LLM接口

In [11]:
class YYFLLM:
    """
    用于调用任何兼容OpenAI接口的服务，并默认使用流式响应。
    """
    def __init__(self, model: str = None, apiKey: str = None, baseUrl: str = None, timeout: int = None):
        """
        初始化客户端。优先使用传入参数，如果未提供，则从环境变量加载。
        """
        self.model = model or os.getenv("LLM_MODEL_ID")
        apiKey = apiKey or os.getenv("LLM_API_KEY")
        baseUrl = baseUrl or os.getenv("LLM_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))
        
        if not all([self.model, apiKey, baseUrl]):
            raise ValueError("模型ID、API密钥和服务地址必须被提供或在.env文件中定义。")

        self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
        """
        调用大语言模型进行思考，并返回其响应。
        """
        print(f"🧠 正在调用 {self.model} 模型...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )
            
            # 处理流式响应
            print("✅ 大语言模型响应成功:")
            collected_content = []
            for chunk in response:
                if not chunk.choices:
                    continue
                content = chunk.choices[0].delta.content or ""
                # print(content, end="", flush=True)
                collected_content.append(content)
            print()  # 在流式输出结束后换行
            return "".join(collected_content)

        except Exception as e:
            print(f"❌ 调用LLM API时发生错误: {e}")
            return None

In [ ]:
try:
    llmClient = YYFLLM()
    
    exampleMessages = [
        {"role": "system", "content": "You are a helpful assistant that writes Python code."},
        {"role": "user", "content": "写一个快速排序算法"}
    ]
    
    print("--- 调用LLM ---")
    responseText = llmClient.think(exampleMessages)
    if responseText:
        print("\n\n--- 完整模型响应 ---")
        print(responseText)

except ValueError as e:
    print(e)


# 工具

## 搜索工具

In [27]:
from serpapi import SerpApiClient

def search(query: str) -> str:
    """
    一个基于SerpApi的实战网页搜索引擎工具。
    它会智能地解析搜索结果，优先返回直接答案或知识图谱信息。
    """
    print(f"🔍 正在执行 [SerpApi] 网页搜索: {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "错误:SERPAPI_API_KEY 未在 .env 文件中配置。"

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",  # 国家代码
            "hl": "zh-cn", # 语言代码
        }
        
        client = SerpApiClient(params)
        results = client.get_dict()

        # 智能解析:优先寻找最直接的答案
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # 如果没有直接答案，则返回前三个有机结果的摘要
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)
        
        return f"对不起，没有找到关于 '{query}' 的信息。"

    except Exception as e:
        return f"搜索时发生错误: {e}"

## 工具执行器

In [40]:
from typing import Dict, Any

class ToolExecutor:
    """
    一个工具执行器，负责管理和执行工具。
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        """
        向工具箱中注册一个新工具。
        """
        if name in self.tools:
            print(f"警告:工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        """
        根据名称获取一个工具的执行函数。
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        获取所有可用工具的格式化描述字符串。
        """
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])


In [30]:
# --- 工具初始化与使用示例 ---
if __name__ == '__main__':
    # 1. 初始化工具执行器
    toolExecutor = ToolExecutor()

    # 2. 注册我们的实战搜索工具
    search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
    toolExecutor.registerTool("Search", search_description, search)
    
    # 3. 打印可用的工具
    print("\n--- 可用的工具 ---")
    print(toolExecutor.getAvailableTools())

    # 4. 智能体的Action调用，这次我们问一个实时性的问题
    print("\n--- 执行 Action: Search['英伟达最新的GPU型号是什么'] ---")
    tool_name = "Search"
    tool_input = "英伟达最新的GPU型号是什么"

    tool_function = toolExecutor.getTool(tool_name)
    if tool_function:
        observation = tool_function(tool_input)
        print("--- 观察 (Observation) ---")
        print(observation)
    else:
        print(f"错误:未找到名为 '{tool_name}' 的工具。")


工具 'Search' 已注册。

--- 可用的工具 ---
- Search: 一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。

--- 执行 Action: Search['英伟达最新的GPU型号是什么'] ---
🔍 正在执行 [SerpApi] 网页搜索: 英伟达最新的GPU型号是什么
--- 观察 (Observation) ---
[1] 比较GeForce 系列最新一代显卡和前代显卡| NVIDIA
比较最新一代RTX 30 系列显卡和前代的RTX 20 系列、GTX 10 和900 系列显卡。查看规格、功能、技术支持等内容。

[2] GeForce RTX 50 系列显卡| NVIDIA
GeForce RTX™ 50 系列GPU 搭载NVIDIA Blackwell 架构，为游戏玩家和创作者带来全新玩法。RTX 50 系列具备强大的AI 算力，带来升级体验和更逼真的画面。

[3] 一文彻底读懂：英伟达GPU分类、架构演进和参数解析
Quadro系列是英伟达专业级GPU产品线，针对商业和专业应用领域进行了优化。常见的产品型号如NVIDIA RTX A6000、A5000等。 Quadro GPU具备强大的计算能力、 ...


In [115]:
weather_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "一个依靠api的查询服务",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "你需要查询的内容"},
            },
            "required": ["query"],
        },
    },
}

model = os.getenv("LLM_MODEL_ID")
apiKey = os.getenv("LLM_API_KEY")
baseUrl = os.getenv("LLM_BASE_URL")
timeout = int(os.getenv("LLM_TIMEOUT", 60))
client = OpenAI(base_url=baseUrl, api_key=apiKey)

# Step 2: 问问题、让 LLM 自己决定要不要调用 tool
resp = client.chat.completions.create(
    model=model,
    max_tokens=512,
    tools=[weather_tool],
    messages=[{"role": "user", "content": "今天上海的天气"}],
)

# === 自我验证 ===
msg = resp.choices[0].message
print("finish_reason:", resp.choices[0].finish_reason)
print("tool_calls:", msg.tool_calls)

assert msg.tool_calls, "预期 LLM 会选择调用 tool（而非直接回答）"
tc = msg.tool_calls[0]
assert tc.function.name == "search", f"预期调用 search、实际 {tc.function.name}"
args = json.loads(tc.function.arguments)
assert args.get("query"), "预期 query 参数有值"
print(f"✅ 练习 1 通过 — {model} 正确选了 search、带 query='{args['query']}' 参数")

finish_reason: tool_calls
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_00_jQdNZUu0tXGqb2q4Av617062', function=Function(arguments='{"query": "上海今天天气"}', name='search'), type='function', index=0)]
✅ 练习 1 通过 — deepseek-v4-flash 正确选了 search、带 query='上海今天天气' 参数


In [43]:
TOOLS = [
    {"type": "function", "function": {"name": "web_search",
        "description": "Search current or external info not in the prompt.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "calculator",
        "description": "Evaluate basic arithmetic with +, -, *, /, parentheses.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
    {"type": "function", "function": {"name": "calendar_lookup",
        "description": "Look up events for a specific date.",
        "parameters": {"type": "object", "properties": {"date": {"type": "string"}}, "required": ["date"]}}},
]

resp = client.chat.completions.create(
    model=model,
    tools=TOOLS,
    messages=[{"role": "user", "content": "What is (19 * 42) - 8?"}],
)

tc = resp.choices[0].message.tool_calls[0]
print(f"LLM 挑了: {tc.function.name}, args: {json.loads(tc.function.arguments)}")

LLM 挑了: calculator, args: {'expression': '(19 * 42) - 8'}


# ReAct

In [31]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

请严格按照以下格式进行回应:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""


In [48]:
import re

class ReActAgent:
    def __init__(self, llm_client: YYFLLM, tool_executor: ToolExecutor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    def run(self, question: str):
        """
        运行ReAct智能体来回答一个问题。
        """
        self.history = [] # 每次运行时重置历史记录
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"--- 第 {current_step} 步 ---")

            # 1. 格式化提示词
            tools_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=tools_desc,
                question=question,
                history=history_str
            )

            # 2. 调用LLM进行思考
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)
            
            print(response_text)
            if not response_text:
                print("错误:LLM未能返回有效响应。")
                break

            # 3. 解析LLM的输出
            thought, action = self._parse_output(response_text)
            
            if thought:
                print(f"思考: {thought}")

            if not action:
                print("警告:未能解析出有效的Action，流程终止。")
                break

            # 4. 执行Action
            if action.startswith("Finish"):
                # 如果是Finish指令，提取最终答案并结束
                final_answer = re.match(r"Finish\[(.*)\]", action).group(1)
                print(f"🎉 最终答案: {final_answer}")
                return final_answer
            
            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                # ... 处理无效Action格式 ...
                continue

            print(f"🎬 行动: {tool_name}[{tool_input}]")
            
            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"错误:未找到名为 '{tool_name}' 的工具。"
            else:
                observation = tool_function(tool_input) # 调用真实工具
            print(f"👀 观察: {observation}")
            
            # 将本轮的Action和Observation添加到历史记录中
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        # 循环结束
        print("已达到最大步数，流程终止。")
        return None

    def _parse_output(self, text: str):
        """解析LLM的输出，提取Thought和Action。
        """
        # Thought: 匹配到 Action: 或文本末尾
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        # Action: 匹配到文本末尾
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    def _parse_action(self, action_text: str):
        """解析Action字符串，提取工具名称和输入。
        """
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        return None, None

In [49]:
# --- 工具初始化与使用示例 ---
if __name__ == '__main__':
    # 1. 初始化工具执行器
    reactagent = ReActAgent(llmClient, toolExecutor, max_steps=5)

    reactagent_input = "华为最新的旗舰手机型号是什么？它的主要卖点是什么？"
    print(f"\n--- 执行 ReAct: {reactagent_input} ---")
    reactagent.run(reactagent_input)


--- 执行 ReAct: 华为最新的旗舰手机型号是什么？它的主要卖点是什么？ ---
--- 第 1 步 ---
🧠 正在调用 deepseek-v4-flash 模型...
✅ 大语言模型响应成功:

Thought: 用户询问华为最新旗舰手机型号及卖点，这需要最新信息，我的知识可能过时，因此需要搜索。
Action: Search[华为最新旗舰手机 2025]
思考: 用户询问华为最新旗舰手机型号及卖点，这需要最新信息，我的知识可能过时，因此需要搜索。
🎬 行动: Search[华为最新旗舰手机 2025]
🔍 正在执行 [SerpApi] 网页搜索: 华为最新旗舰手机 2025
👀 观察: [1] 华为手机- 华为官网
智能手机 ; Pura 系列. 先锋影像. HUAWEI Pura 90 Pro Max. ￥6499 起 ; Mate 系列. 非凡旗舰. HUAWEI Mate 80 Pro Max 风驰版. ￥8499 起 ; Pocket 系列. 美学新篇. HUAWEI ...

[2] 2026年华为手机各系列介绍及选购指南（8月份更新）暑期 ...
2026年华为手机各系列介绍及选购指南（8月份更新）暑期华为手机推荐 · 一、华为Mate系列（5000元以上） · 二、华为Pura系列（4000元以上） · 三、华为nova系列（2000-4000元）.

[3] 华为发布会- 华为官网
2025.11.25华为Mate 80 系列| Mate X7 及全场景新品发布会华为全新发布Mate 80 系列、Mate X7、MatePad Edge、WATCH Ultimate 2、华为智慧屏MateTV Max、华为路由X3 Pro ...
--- 第 2 步 ---
🧠 正在调用 deepseek-v4-flash 模型...
✅ 大语言模型响应成功:

Thought: 历史结果已显示华为最新旗舰包括 Mate 80 系列和 Pura 系列，但要回答“最新旗舰手机型号及主要卖点”，还需要进一步确认 Mate 80 系列的旗舰型号和核心功能。
Action: Search[华为Mate 80 Pro Max 主要卖点]
思考: 历史结果已显示华为最新旗舰包括 Mate 80 系列和 Pura 系列，但要回答“最新旗舰手机型号及主要卖点”，还需要

## 自写一个ReAct

In [ ]:
def tool_calculator(expression: str) -> str:
    """安全的計算器：只允許 + - * / 跟數字。"""
    allowed = set("0123456789.+-*/() ")
    if any(c not in allowed for c in expression):
        return f"error: 表達式含不允許字元（{expression}）"
    try:
        return str(eval(expression))  # noqa: S307 — 已用 whitelist
    except Exception as e:  # noqa: BLE001
        return f"error: {e}"

def tool_lookup(query: str) -> str:
    facts={
        '上海人口数': '1000000',
        '上海高楼数': '5000',
        '我的成绩': '100'
    }
    return facts.get(query.strip(), f"unknown, {query}")

TOOLS = [
    {
        "type": 'function',
        "function": {
            "name": "calculator",
            "description": "简单的加减乘除，字符串输入",
            "parameters": {
                "type": "object",
                "properties":{
                    "expression": {"type": "string", "description": "算数的表达式"}
                },
                "required": ['expression']
            }
        }
    },
    {
        "type": 'function',
        "function": {
            "name": "lookup",
            "description": "事实查找",
            "parameters": {
                "type": "object",
                "properties":{
                "query": {"type": "string", "description": "查询关键字"}
                },
                "required": ["query"]
            }
        }
    }
]

TOOLS_IMPL = {
    "calculator": lambda x: tool_calculator(x['expression']),
    "lookup": lambda x: tool_lookup(x['query'])
}

def react_loop(question: str, max_iter: int = 6, client: Any = None) ->dict:
    if not client:
        apiKey = os.getenv("LLM_API_KEY")
        baseUrl = os.getenv("LLM_BASE_URL")
        timeout = int(os.getenv("LLM_TIMEOUT", 60))
        client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    messages = [{'role': 'user', 'content': question}]
    trace = []
    model = os.getenv("LLM_MODEL_ID")

    for step in range(max_iter):
        resp = client.chat.completions.create(
            model=model,
            tools=TOOLS,
            messages=messages
        )
        msg = resp.choices[0].message
        content = msg.content or ""
        tool_calls = msg.tool_calls or []

        assistant_entry = {"role": 'assistant', 'content': content}
        if tool_calls:
            assistant_entry['tool_calls'] = [
                {
                    'id': tc.id,
                    'type': 'function',
                    'function': {
                        'arguments': tc.function.arguments,
                        'name': tc.function.name
                    }
                }
                for tc in tool_calls
            ]
        messages.append(assistant_entry)

        if resp.choices[0].finish_reason == 'stop' or not tool_calls:
            trace.append({
                'step': step,
                'thought': content,
                'tool': None,
                'obs': None
            })
            return {'final': content, 'trace': trace, 'steps': step + 1}

        last_obs=""
        for tc in tool_calls:
            fn = TOOLS_IMPL.get(tc.function.name)
            args = json.loads(tc.function.arguments)
            obs = fn(args) if fn else f"error: unknown tool {tc.function.name}"
            last_obs = obs
            messages.append({
                'role': 'tool',
                'tool_call_id': tc.id,
                'content': obs
            })

            trace.append({
                'step': step,
                'thought': content,
                'tool': tc.function.name,
                'tool_input': args,
                'obs': last_obs
            })
    return {'final': None, 'trace': trace, 'steps': max_iter, 'truncated': True}

In [112]:
if __name__ == "__main__":
    question = "'上海人口数' 除以 '上海高楼数'、答案保留 4 位小數。"
    print("-" * 60)

    result = react_loop(question, max_iter=5)

    for entry in result["trace"]:
        print(f"[step {entry['step']}] thought: {(entry['thought'] or '')[:80]}...")
        if entry["tool"]:
            print(f"           tool: {entry['tool']}({entry.get('tool_input')}) → {entry['obs']}")
    print("-" * 60)
    print(f"✅ 最終答案：{result['final']}")
    print(f"   共 {result['steps']} 輪")

    # 寬鬆驗證（小 model 不一定精確到 4 位小數）
    assert result.get("final") is not None or result.get("truncated"), "loop 應收尾或顯式 truncate"
    print("✅ 練習 3 通過 — 你已用本機 qwen2.5:3b 跑通 ReAct + tool use、$0/run")

------------------------------------------------------------
ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_zTk2gt3AICWv5MkKYSC10959', function=Function(arguments='{"query": "上海人口数"}', name='lookup'), type='function', index=0), ChatCompletionMessageFunctionToolCall(id='call_01_pGaj3qFtcl22J7Cd8YjN9663', function=Function(arguments='{"query": "上海高楼数"}', name='lookup'), type='function', index=1)], reasoning_content="The user wants me to divide Shanghai's population by Shanghai's number of tall buildings, keeping 4 decimal places.\n\nI need to look up facts. Let me use the lookup tool.")
ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_NQ1kqZlqjmTE2c1aNPFv2883', function=Function(arguments='{"expression": "1000000/5000"}', name='calc

# Planning

## 规划器

In [5]:
PLANNER_PROMPT_TEMPLATE = """
你是一个顶级的AI规划专家。你的任务是将用户提出的复杂问题分解成一个由多个简单步骤组成的行动计划。
请确保计划中的每个步骤都是一个独立的、可执行的子任务，并且严格按照逻辑顺序排列。
你的输出必须是一个Python列表，其中每个元素都是一个描述子任务的字符串。

问题: {question}

请严格按照以下格式输出你的计划,```python与```作为前后缀是必要的:
```python
["步骤1", "步骤2", "步骤3", ...]
```
"""

In [6]:
import ast

class Planner:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def plan(self, question: str) -> list[str]:
        """
        根据用户问题生成一个行动计划。
        """
        prompt = PLANNER_PROMPT_TEMPLATE.format(question=question)
        
        # 为了生成计划，我们构建一个简单的消息列表
        messages = [{"role": "user", "content": prompt}]
        
        print("--- 正在生成计划 ---")
        # 使用流式输出来获取完整的计划
        response_text = self.llm_client.think(messages=messages) or ""
        
        print(f"✅ 计划已生成:\n{response_text}")
        
        # 解析LLM输出的列表字符串
        try:
            # 找到```python和```之间的内容
            plan_str = response_text.split("```python")[1].split("```")[0].strip()
            # 使用ast.literal_eval来安全地执行字符串，将其转换为Python列表
            plan = ast.literal_eval(plan_str)
            return plan if isinstance(plan, list) else []
        except (ValueError, SyntaxError, IndexError) as e:
            print(f"❌ 解析计划时出错: {e}")
            print(f"原始响应: {response_text}")
            return []
        except Exception as e:
            print(f"❌ 解析计划时发生未知错误: {e}")
            return []


## 执行器

In [7]:
EXECUTOR_PROMPT_TEMPLATE = """
你是一位顶级的AI执行专家。你的任务是严格按照给定的计划，一步步地解决问题。
你将收到原始问题、完整的计划、以及到目前为止已经完成的步骤和结果。
请你专注于解决“当前步骤”，并仅输出该步骤的最终答案，不要输出任何额外的解释或对话。

# 原始问题:
{question}

# 完整计划:
{plan}

# 历史步骤与结果:
{history}

# 当前步骤:
{current_step}

请仅输出针对“当前步骤”的回答:
"""

In [8]:
class Executor:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def execute(self, question: str, plan: list[str]) -> str:
        """
        根据计划，逐步执行并解决问题。
        """
        history = "" # 用于存储历史步骤和结果的字符串
        
        print("\n--- 正在执行计划 ---")
        
        for i, step in enumerate(plan):
            print(f"\n-> 正在执行步骤 {i+1}/{len(plan)}: {step}")
            
            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                question=question,
                plan=plan,
                history=history if history else "无", # 如果是第一步，则历史为空
                current_step=step
            )
            
            messages = [{"role": "user", "content": prompt}]
            
            response_text = self.llm_client.think(messages=messages) or ""
            
            # 更新历史记录，为下一步做准备
            history += f"步骤 {i+1}: {step}\n结果: {response_text}\n\n"
            
            print(f"✅ 步骤 {i+1} 已完成，结果: {response_text}")

        # 循环结束后，最后一步的响应就是最终答案
        final_answer = response_text
        return final_answer


## 整合器

In [9]:
class PlanAndSolveAgent:
    def __init__(self, llm_client):
        """
        初始化智能体，同时创建规划器和执行器实例。
        """
        self.llm_client = llm_client
        self.planner = Planner(self.llm_client)
        self.executor = Executor(self.llm_client)

    def run(self, question: str):
        """
        运行智能体的完整流程:先规划，后执行。
        """
        print(f"\n--- 开始处理问题 ---\n问题: {question}")
        
        # 1. 调用规划器生成计划
        plan = self.planner.plan(question)
        
        # 检查计划是否成功生成
        if not plan:
            print("\n--- 任务终止 --- \n无法生成有效的行动计划。")
            return

        # 2. 调用执行器执行计划
        final_answer = self.executor.execute(question, plan)
        
        print(f"\n--- 任务完成 ---\n最终答案: {final_answer}")


In [15]:
if __name__ == '__main__':
    try:
        llm_client = YYFLLM()
        agent = PlanAndSolveAgent(llm_client)
        question = "你是一家跨境电商公司的运营负责人。公司主营一款智能家居小配件（单价$49），过去一年销量稳定，但最近连续3个月，北美市场的退货率从原来的5%急剧上升至15%。老板要求你在下周一的例会上提交一份可执行的整改方案，目标是在下个季度将退货率降回8%以内，同时不能增加超过5%的运营总成本。"
        agent.run(question)
    except ValueError as e:
        print(e)


--- 开始处理问题 ---
问题: 你是一家跨境电商公司的运营负责人。公司主营一款智能家居小配件（单价$49），过去一年销量稳定，但最近连续3个月，北美市场的退货率从原来的5%急剧上升至15%。老板要求你在下周一的例会上提交一份可执行的整改方案，目标是在下个季度将退货率降回8%以内，同时不能增加超过5%的运营总成本。
--- 正在生成计划 ---
🧠 正在调用 deepseek-v4-flash 模型...
✅ 大语言模型响应成功:

✅ 计划已生成:
```python
[
    "步骤1：拉取近6个月北美退货订单明细，按SKU、渠道、仓库、承运商、州和退货原因分类，计算各维度退货率并锁定退货率异常集中的环节。",
    "步骤2：选取近30-60天至少50个退货用户做问卷或电话回访，结合退货备注、客服聊天记录和平台差评，定位退货率飙升的直接触发因素。",
    "步骤3：核查最近3个月内产品设计/供应商批次、包装SOP、listing页面、广告投放、促销流量和平台退货政策的变化，排除内部变动或外部规则导致的系统性问题。",
    "步骤4：对退回实物进行开箱检测并录像，记录功能故障、外观损坏、配件缺失、包装挤压、误购/不会使用等情况，量化质量缺陷、物流损伤和说明不清的占比。",
    "步骤5：汇总数据形成根因结论，按“影响大且成本低”优先排序；对每项整改措施做成本和收益预估，确保所有新措施合计带来的运营总成本增量不超过现有运营总成本的5%。",
    "步骤6：若存在批次质量问题，立即与供应商联合复盘，锁定不良批次范围，增加出厂抽检或更换关键物料，并制定在途库存、安全库存和不良品处置方案。",
    "步骤7：若存在listing描述、参数或使用说明与实际不符，48小时内修正北美站标题、图片、规格、兼容性列表、安装说明和FAQ，减少因预期错位导致的误购退货。",
    "步骤8：在商品详情页增加短视频、兼容性检查工具、注意事项和售前自动问答；售前客服主动确认使用场景，使不匹配客户在下单前就被识别。",
    "步骤9：针对物流损坏高发的仓库/承运商/州，先用前3个重点区域试点改进缓冲包装或更换配送物流，用试点结果验证后再推广。",
    "步骤10：优化退货流程：退货原因改为必选下拉框并强制上传照片/视频；退回商品按可售/不可售分拣，可

# Reflection

## 记忆模块

In [17]:
from typing import List, Dict, Any, Optional

class Memory:
    """
    一个简单的短期记忆模块，用于存储智能体的行动与反思轨迹。
    """

    def __init__(self):
        """
        初始化一个空列表来存储所有记录。
        """
        self.records: List[Dict[str, Any]] = []

    def add_record(self, record_type: str, content: str):
        """
        向记忆中添加一条新记录。

        参数:
        - record_type (str): 记录的类型 ('execution' 或 'reflection')。
        - content (str): 记录的具体内容 (例如，生成的代码或反思的反馈)。
        """
        record = {"type": record_type, "content": content}
        self.records.append(record)
        print(f"📝 记忆已更新，新增一条 '{record_type}' 记录。")

    def get_trajectory(self) -> str:
        """
        将所有记忆记录格式化为一个连贯的字符串文本，用于构建提示词。
        """
        trajectory_parts = []
        for record in self.records:
            if record['type'] == 'execution':
                trajectory_parts.append(f"--- 上一轮尝试 (代码) ---\n{record['content']}")
            elif record['type'] == 'reflection':
                trajectory_parts.append(f"--- 评审员反馈 ---\n{record['content']}")
        
        return "\n\n".join(trajectory_parts)

    def get_last_execution(self) -> Optional[str]:
        """
        获取最近一次的执行结果 (例如，最新生成的代码)。
        如果不存在，则返回 None。
        """
        for record in reversed(self.records):
            if record['type'] == 'execution':
                return record['content']
        return None


In [18]:
# 初始提示词
INITIAL_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。请根据以下要求，编写一个Python函数。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。

要求: {task}

请直接输出代码，不要包含任何额外的解释。
"""

# 反思提示词
REFLECT_PROMPT_TEMPLATE = """
你是一位极其严格的代码评审专家和资深算法工程师，对代码的性能有极致的要求。
你的任务是审查以下Python代码，并专注于找出其在<strong>算法效率</strong>上的主要瓶颈。

# 原始任务:
{task}

# 待审查的代码:
```python
{code}
```

请分析该代码的时间复杂度，并思考是否存在一种<strong>算法上更优</strong>的解决方案来显著提升性能。
如果存在，请清晰地指出当前算法的不足，并提出具体的、可行的改进算法建议（例如，使用筛法替代试除法）。
如果代码在算法层面已经达到最优，才能回答“无需改进”。

请直接输出你的反馈，不要包含任何额外的解释。
"""

# 优化提示词

REFINE_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。你正在根据一位代码评审专家的反馈来优化你的代码。

# 原始任务:
{task}

# 你上一轮尝试的代码:
{last_code_attempt}
评审员的反馈：
{feedback}

请根据评审员的反馈，生成一个优化后的新版本代码。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。
请直接输出优化后的代码，不要包含任何额外的解释。
"""


In [ ]:
class ReflectionAgent:
    def __init__(self, llm_client, max_iterations=3):
        self.llm_client = llm_client
        self.memory = Memory()
        self.max_iterations = max_iterations

    def run(self, task: str):
        print(f"\n--- 开始处理任务 ---\n任务: {task}")

        # --- 1. 初始执行 ---
        print("\n--- 正在进行初始尝试 ---")
        initial_prompt = INITIAL_PROMPT_TEMPLATE.format(task=task)
        initial_code = self._get_llm_response(initial_prompt)
        self.memory.add_record("execution", initial_code)
        print(initial_code)

        # --- 2. 迭代循环:反思与优化 ---
        for i in range(self.max_iterations):
            print(f"\n--- 第 {i+1}/{self.max_iterations} 轮迭代 ---")

            # a. 反思
            print("\n-> 正在进行反思...")
            last_code = self.memory.get_last_execution()
            reflect_prompt = REFLECT_PROMPT_TEMPLATE.format(task=task, code=last_code)
            feedback = self._get_llm_response(reflect_prompt)
            self.memory.add_record("reflection", feedback)

            # b. 检查是否需要停止
            if "无需改进" in feedback:
                print("\n✅ 反思认为代码已无需改进，任务完成。")
                break

            # c. 优化
            print("\n-> 正在进行优化...")
            refine_prompt = REFINE_PROMPT_TEMPLATE.format(
                task=task,
                last_code_attempt=last_code,
                feedback=feedback
            )
            refined_code = self._get_llm_response(refine_prompt)
            self.memory.add_record("execution", refined_code)
            print(refined_code)
        
        final_code = self.memory.get_last_execution()
        print(f"\n--- 任务完成 ---\n最终生成的代码:\n{final_code}\n```")
        return final_code

    def _get_llm_response(self, prompt: str) -> str:
        """一个辅助方法，用于调用LLM并获取完整的流式响应。"""
        messages = [{"role": "user", "content": prompt}]
        response_text = self.llm_client.think(messages=messages) or ""
        return response_text


In [22]:
if __name__ == '__main__':
    # 1. 初始化LLM客户端 (请确保你的 .env 和 llm_client.py 文件配置正确)
    try:
        llm_client = YYFLLM()
    except Exception as e:
        print(f"初始化LLM客户端时出错: {e}")
        exit()

    # 2. 初始化 Reflection 智能体，设置最多迭代2轮
    agent = ReflectionAgent(llm_client, max_iterations=2)

    # 3. 定义任务并运行智能体
    task = "编写一个Python函数，找出1到n之间所有的素数 (prime numbers)。"
    agent.run(task)


--- 开始处理任务 ---
任务: 编写一个Python函数，找出1到n之间所有的素数 (prime numbers)。

--- 正在进行初始尝试 ---
🧠 正在调用 deepseek-v4-flash 模型...
✅ 大语言模型响应成功:

📝 记忆已更新，新增一条 'execution' 记录。
```python
def find_primes_up_to(n: int) -> list[int]:
    """
    Return a list of all prime numbers between 1 and n (inclusive).

    Args:
        n: The upper bound (inclusive). Must be a non-negative integer.

    Returns:
        A list of prime numbers from 1 to n, in ascending order.
        Returns an empty list if n is less than 2.
    """
    if n < 2:
        return []

    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False

    for i in range(2, int(n ** 0.5) + 1):
        if is_prime[i]:
            for j in range(i * i, n + 1, i):
                is_prime[j] = False

    return [num for num, prime in enumerate(is_prime) if prime]
```

--- 第 1/2 轮迭代 ---

-> 正在进行反思...
🧠 正在调用 deepseek-v4-flash 模型...
✅ 大语言模型响应成功:

📝 记忆已更新，新增一条 'reflection' 记录。

✅ 反思认为代码已无需改进，任务完成。

--- 任务完成 ---
最终生成的代码:
```python
```python
de